---
<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Dinámica del clima   
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:Blue; font-size:160%;">Unidad 4:El Clima de México y su variabilidad  </span><br/>
  <span style="color:Blue; font-size:140%;">Facultad de Ciencias  |  Semestre 2027-I</span><br/>
</p>

---

# **<font color="Navy">   El ciclo anual del clima mexicano </font>**
---
---


**Objetivo general.** Caracterizar y explicar, con datos observados y de reanálisis, el ciclo anual de la precipitación, la temperatura y la circulación atmosférica sobre México, y relacionarlo con sus forzantes dinámicos. 

In [ ]:
ARCHIVO   = "/home/jovyan/Climatologia_mexico/data/skin-temp_1981-2024.nc"      # ruta al NetCDF (admite comodines: "datos_*.nc")
VARIABLE  = None                # p. ej. "t2m"; None = detectar automáticamente
SELECCION = {}                  # dimensiones extra, p. ej. {"level": 850}
INI, FIN  = 1994, 2020          # recorte del periodo (años), p. ej. 1991, 2020
BASE_INI, BASE_FIN = None, None # periodo base de climatología/anomalías; None = todo el periodo
TRIMESTRES = "estaciones"       # "estaciones" (DJF, MAM, JJA, SON) o "moviles" (12 trimestres)
ANIO_ANOMALIA = None            # año para anomalías mensuales/trimestrales; None = último año completo
ANIOS_PANEL   = None            # lista de años para el panel de anomalías anuales; None = últimos 12
CMAP      = "RdYlBu_r"          # paleta para temperatura
CMAP_ANOM = "RdBu_r"            # paleta divergente para anomalías
SALIDA    = "resultados"        # carpeta de salida

In [ ]:
!pip install cartopy xarray netCDF4 h5netcdf

In [ ]:
import xarray as xr
import numpy as np 
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
#from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.dates as mdates
import glob, os, warnings

plt.rcParams.update({"figure.dpi": 110})
#os.makedirs(SALIDA, exist_ok=True)

MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio",
         "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
ABR = [m[:3] for m in MESES]
NOMBRES_ESTACIONES = {12: "DJF", 3: "MAM", 6: "JJA", 9: "SON"}
INICIOS = (12, 3, 6, 9) if TRIMESTRES == "estaciones" else tuple(range(1, 13))   # mes de inicio de cada trimestre

In [ ]:
ds = xr.open_dataset(ARCHIVO, engine='netcdf4')
ds

In [ ]:
# Primero vamos a convertir los datos de Kelvin a Celcius

ds["skt"] = ds["skt"]-273.15

## <font color = "DarkBlue"> 1. Promedio anual </font>

Se promedia cada año y se calcula el **promedio anual climatológico** como la media de las 12 climatologías mensuales.

In [ ]:
# Antes de calcular el promedio anual, primero necesitamos la climatolog[ia mensual

# climatología mensual (dim 'month' = 1..12) y promedio anual climatológico
clim_m = _________________________
media_anual = ____________________
media_anual.attrs.update(units="°C", long_name="Promedio anual de temperatura")

In [ ]:
# promedio de cada año (solo años completos)
cuenta = ds["skt"].valid_time.to_series().groupby(ds["skt"].valid_time.dt.year.values).count()
anios_completos = cuenta[cuenta == 12].index.values
anual = ds["skt"].groupby("valid_time.year").mean("valid_time").sel(year=anios_completos)

#Calculo de la temperatura promedio anual de todo Mexico:
peso_lat = np.cos(np.deg2rad(ds["skt"].latitude))   
print(f"Promedio anual regional: {float(media_anual.weighted(peso_lat).mean(('latitude', 'longitude'))):.2f} °C "
      f"(mín {float(media_anual.min()):.1f} °C, máx {float(media_anual.max()):.1f} °C)")

In [ ]:
t0, t1 = str(ds["skt"].valid_time.values[0])[:7], str(ds["skt"].valid_time.values[-1])[:7]
periodo = f"{t0[:4]}–{t1[:4]}"
periodo

In [ ]:
# --- mapa del promedio anual con barra de colores de 2 en 2
titulo = f"Temperatura · promedio anual ({periodo})"
campo = media_anual                      # 2D (latitude, longitude)
lon, lat = campo.longitude.values, campo.latitude.values

# paso medio de la malla (0.5 si solo hay un punto),]
dx = np.r_[np.abs(np.diff(lon)), 0.5][:max(lon.size - 1, 1)].mean()
dy = np.r_[np.abs(np.diff(lat)), 0.5][:max(lat.size - 1, 1)].mean()

# extensión del mapa y tamaño de la figura
ext = [lon.min() - dx / 2, lon.max() + dx / 2, lat.min() - dy / 2, lat.max() + dy / 2]
ar = (ext[1] - ext[0]) / max(ext[3] - ext[2], 1e-6)
w = float(np.clip(3.2 * ar, 2.6, 5.5))

# niveles cada 2 °C que cubren todos los datos
paso = 3
niveles = np.arange(np.floor(float(campo.min()) / paso) * paso,
                    np.ceil(float(campo.max()) / paso) * paso + paso,
                    paso)

fig, ax = plt.subplots(figsize=(w + 1.5, 3.1 + 1.2),
                       subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent(ext, crs=ccrs.PlateCarree())
ax.coastlines(linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.5)
gl.top_labels = gl.right_labels = False
gl.xlabel_style = gl.ylabel_style = {"size": 7}

campo.plot.pcolormesh(
    ax=ax, x="longitude", y="latitude",
    cmap=CMAP, levels=niveles, shading="auto",
    transform=ccrs.PlateCarree(),
    cbar_kwargs={"label": "Temperatura (°C)", "shrink": 0.8, "pad": 0.02, "ticks": niveles},
)

ax.set_title(titulo, fontsize=9)
fig.suptitle(titulo, fontsize=13, y=0.995)
plt.show()

In [ ]:
# --- serie de promedios anuales (media regional)
serie_anual = anual.weighted(peso_lat).mean(("latitude", "longitude"))
fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(serie_anual.year, serie_anual.values, "o-", ms=3)
ax.set_xlabel("Año"); ax.set_ylabel("Temperatura media regional (°C)")
ax.set_title("Promedio anual (media espacial ponderada)"); ax.grid(alpha=0.3)
plt.show()

## <font color = "DarkBlue"> 1. Climatologías mensuales </font>

Ahora vamos a ver cómo es la climatología de la temperatura mensual 

In [ ]:
# --- mapa mensual (12 paneles) con barra de colores de 2 en 2
campos, ncol = clim_m, 3
dim = campos.dims[0]
titulo = f"Climatología mensual de temperatura ({periodo})"

lon, lat = campos.longitude.values, campos.latitude.values
ar = float(np.clip((np.ptp(lon) / max(np.ptp(lat), 1e-6)), 0.8, 1.8))

paso = 3
niveles = np.arange(np.floor(float(campos.min()) / paso) * paso,
                    np.ceil(float(campos.max()) / paso) * paso + paso,
                    paso)                                  # múltiplos de 2 que cubren todos los datos

g = campos.assign_coords({dim: MESES}).plot.pcolormesh(
    x="longitude", y="latitude", col=dim, col_wrap=ncol,
    cmap=CMAP, levels=niveles, shading="auto",
    size=3.1, aspect=ar,
    subplot_kws={"projection": ccrs.PlateCarree()},
    transform=ccrs.PlateCarree(),
    cbar_kwargs={"label": "Temperatura (°C)", "shrink": 0.8, "pad": 0.02, "ticks": niveles},
)

np.frompyfunc(GeoAxes.coastlines, 1, 1)(g.axs)
np.frompyfunc(GeoAxes.add_feature, 2, 1)(g.axs, np.array(cfeature.BORDERS, dtype=object))

g.set_titles("{value}", size=9)
g.fig.suptitle(titulo, fontsize=13, y=1.02)
plt.show()

## <font color = "DarkBlue"> Climatologías trimestrales </font>
`TRIMESTRES = "estaciones"` → DJF, MAM, JJA, SON (4 mapas). 
`TRIMESTRES = "moviles"` → Ene-Feb-Mar, Feb-Mar-Abr, … (12 mapas). 
Cada trimestre es el promedio de 3 climatologías mensuales consecutivas.

In [ ]:
campos_t, etiq_t = [], []
for ini in INICIOS:
    idx = [(ini - 1 + k) % 12 for k in range(3)]                  # posiciones de los 3 meses (0..11)
    campos_t.append(clim_m.isel(month=idx).mean("month"))
    nombre = "-".join(ABR[i] for i in idx)
    etiq_t.append(f"{NOMBRES_ESTACIONES[ini]} ({nombre})" if TRIMESTRES == "estaciones" else nombre)
clim_t = xr.concat(campos_t, dim="trimestre").assign_coords(trimestre=etiq_t)
clim_t.attrs["units"] = "°C"

In [ ]:
paneles = [(clim_t, etiq_t, f"Climatología trimestral de temperatura ({periodo})",
            os.path.join(SALIDA, "04_climatologia_trimestral.png"),
            2 if TRIMESTRES == "estaciones" else 4, CMAP, False, "Temperatura (°C)")]

for (campos, etiquetas, titulo, archivo, ncol, cmap, simetrico, etiqueta) in paneles:
    n = campos.shape[0]
    nfil = int(np.ceil(n / ncol))
    # extensión del mapa y tamaño de la figura
    dx = float(np.abs(np.diff(campos.longitude.values)).mean()) if campos.sizes["longitude"] > 1 else 0.5
    dy = float(np.abs(np.diff(campos.latitude.values)).mean()) if campos.sizes["latitude"] > 1 else 0.5
    ext = [float(campos.longitude.min()) - dx / 2, float(campos.longitude.max()) + dx / 2,
           float(campos.latitude.min()) - dy / 2, float(campos.latitude.max()) + dy / 2]
    ar = (ext[1] - ext[0]) / max(ext[3] - ext[2], 1e-6)
    w = float(np.clip(3.2 * ar, 2.6, 5.5))
    # escala de color común a todos los paneles
    if simetrico:
        m = float(np.nanquantile(np.abs(campos.values), 0.99)); m = m if m > 0 else 1.0
        vmin, vmax = -m, m
    else:
        vmin, vmax = float(campos.quantile(0.01)), float(campos.quantile(0.99))
    kw = {"subplot_kw": {"projection": ccrs.PlateCarree()}} if True else {}
    fig, axs = plt.subplots(nfil, ncol, figsize=(w * ncol + 1.5, 3.1 * nfil + 1.2), squeeze=False, **kw)
    for k in range(nfil * ncol):
        i, j = divmod(k, ncol)
        ax = axs[i, j]
        if k >= n:
            ax.set_visible(False)
            continue
        if True:
            ax.set_extent(ext, crs=ccrs.PlateCarree())
            ax.coastlines(linewidth=0.6); ax.add_feature(cfeature.BORDERS, linewidth=0.4)
            gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.5)
            gl.top_labels = gl.right_labels = False
            gl.left_labels = (j == 0); gl.bottom_labels = (i == nfil - 1)
            gl.xlabel_style = gl.ylabel_style = {"size": 7}
            tr = {"transform": ccrs.PlateCarree()}
        else:
            ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.tick_params(labelsize=7); tr = {}
        campo = campos.isel({campos.dims[0]: k})
        pm = ax.pcolormesh(campo.longitude, campo.latitude, campo.values, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto", **tr)
        ax.set_title(etiquetas[k], fontsize=9)
    fig.suptitle(titulo, fontsize=13, y=0.995)
    fig.colorbar(pm, ax=axs.ravel().tolist(), shrink=0.8, pad=0.02, label=etiqueta)
    plt.show()

## <font color= "DarkBlue"> Anomalías </font>

Una **anomalía** es el valor observado menos la climatología del **periodo base** (`BASE_INI`–`BASE_FIN`; con `None` se usa todo el periodo). Los mapas usan una paleta divergente con **escala simétrica** respecto de 0 (azul = más frío que lo normal, rojo = más cálido).

### Anomalías mensuales y series regionales

In [ ]:
# --- climatología del periodo base y anomalías mensuales
base = ds["skt"].sel(valid_time=slice(str(BASE_INI) if BASE_INI else None, str(BASE_FIN) if BASE_FIN else None))

clim_base = base.groupby("valid_time.month").mean("valid_time")
anom_m = ds["skt"].groupby("valid_time.month") - clim_base                 # (time, lat, lon)
anom_m.attrs["units"] = "°C"
base_txt = f"{BASE_INI or t0[:4]}–{BASE_FIN or t1[:4]}"
print("Periodo base:", base_txt)

In [ ]:
# --- anomalías anuales
anom_a = anual - clim_base.mean("month")                           # (year, lat, lon)
anom_a.attrs["units"] = "°C"

In [ ]:
# --- series regionales
anom_reg = anom_m.weighted(peso_lat).mean(("latitude", "longitude"))
media_movil = anom_reg.rolling(valid_time=12, center=True, min_periods=12).mean()

fig, axs = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={"width_ratios": [2, 1]})
axs[0].bar(anom_reg.valid_time.values, anom_reg.values, width=25,
           color=np.where(anom_reg.values >= 0, "tab:red", "tab:blue"), alpha=0.6)
axs[0].plot(media_movil.valid_time.values, media_movil.values, "k-", lw=1.5, label="Media móvil 12 meses")
axs[0].axhline(0, color="k", lw=0.6); axs[0].legend(); axs[0].set_ylabel("Anomalía (°C)")
axs[0].set_title(f"Anomalía mensual de temperatura, media regional (base {base_txt})")
if anom_a.sizes["year"] >= 1:
    anom_a_reg = anom_a.weighted(peso_lat).mean(("latitude", "longitude"))
    axs[1].bar(anom_a_reg.year.values, anom_a_reg.values, color=np.where(anom_a_reg.values >= 0, "tab:red", "tab:blue"))
    axs[1].axhline(0, color="k", lw=0.6); axs[1].set_title("Anomalía anual (media regional)")
fig.tight_layout() 
plt.show()

### Observemos como fueron los ultimos años:

In [ ]:
anios_disp = [int(y) for y in anom_a.year.values]
anios_panel = [y for y in (ANIOS_PANEL or anios_disp[-12:]) if y in anios_disp]
if anios_panel:
    paneles = [(anom_a.sel(year=anios_panel), [str(y) for y in anios_panel],
                f"Anomalía anual de temperatura (base {base_txt})",
                os.path.join(SALIDA, "06_anomalias_anuales.png"), 4, CMAP_ANOM, True, "Anomalía (°C)")]
    for (campos, etiquetas, titulo, archivo, ncol, cmap, simetrico, etiqueta) in paneles:
        n = campos.shape[0]
        nfil = int(np.ceil(n / ncol))
        # extensión del mapa y tamaño de la figura
        dx = float(np.abs(np.diff(campos.longitude.values)).mean()) if campos.sizes["longitude"] > 1 else 0.5
        dy = float(np.abs(np.diff(campos.latitude.values)).mean()) if campos.sizes["latitude"] > 1 else 0.5
        ext = [float(campos.longitude.min()) - dx / 2, float(campos.longitude.max()) + dx / 2,
               float(campos.latitude.min()) - dy / 2, float(campos.latitude.max()) + dy / 2]
        ar = (ext[1] - ext[0]) / max(ext[3] - ext[2], 1e-6)
        w = float(np.clip(3.2 * ar, 2.6, 5.5))
        # escala de color común a todos los paneles
        if simetrico:
            m = float(np.nanquantile(np.abs(campos.values), 0.99)); m = m if m > 0 else 1.0
            vmin, vmax = -m, m
        else:
            vmin, vmax = float(campos.quantile(0.01)), float(campos.quantile(0.99))
        kw = {"subplot_kw": {"projection": ccrs.PlateCarree()}} if True else {}
        fig, axs = plt.subplots(nfil, ncol, figsize=(w * ncol + 1.5, 3.1 * nfil + 1.2), squeeze=False, **kw)
        for k in range(nfil * ncol):
            i, j = divmod(k, ncol)
            ax = axs[i, j]
            if k >= n:
                ax.set_visible(False)
                continue
            if True:
                ax.set_extent(ext, crs=ccrs.PlateCarree())
                ax.coastlines(linewidth=0.6); ax.add_feature(cfeature.BORDERS, linewidth=0.4)
                gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.5)
                gl.top_labels = gl.right_labels = False
                gl.left_labels = (j == 0); gl.bottom_labels = (i == nfil - 1)
                gl.xlabel_style = gl.ylabel_style = {"size": 7}
                tr = {"transform": ccrs.PlateCarree()}
            else:
                ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.tick_params(labelsize=7); tr = {}
            campo = campos.isel({campos.dims[0]: k})
            pm = ax.pcolormesh(campo.longitude, campo.latitude, campo.values, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto", **tr)
            ax.set_title(etiquetas[k], fontsize=9)
        fig.suptitle(titulo, fontsize=13, y=0.995)
        fig.colorbar(pm, ax=axs.ravel().tolist(), shrink=0.8, pad=0.02, label=etiqueta)
        plt.show()
else:
    print("No hay años completos para mostrar anomalías anuales.")

### Ahora veamos las anomalías mensuales de un año en particular.

In [ ]:
anio_an = ANIO_ANOMALIA or (anios_disp[-1] if anios_disp else int(ds["skt"].valid_time.dt.year.values[-1]))
sub = anom_m.sel(valid_time=str(anio_an))
etiq_m = [MESES[int(m) - 1] for m in sub.valid_time.dt.month.values]
print("Año de análisis:", anio_an, "| meses disponibles:", len(etiq_m))
paneles = [(sub, etiq_m, f"Anomalías mensuales de temperatura · {anio_an} (base {base_txt})", 4, CMAP_ANOM, True, "Anomalía (°C)")]

for (campos, etiquetas, titulo, ncol, cmap, simetrico, etiqueta) in paneles:
    n = campos.shape[0]
    nfil = int(np.ceil(n / ncol))
    # extensión del mapa y tamaño de la figura
    dx = float(np.abs(np.diff(campos.longitude.values)).mean()) if campos.sizes["longitude"] > 1 else 0.5
    dy = float(np.abs(np.diff(campos.latitude.values)).mean()) if campos.sizes["latitude"] > 1 else 0.5
    ext = [float(campos.longitude.min()) - dx / 2, float(campos.longitude.max()) + dx / 2,
           float(campos.latitude.min()) - dy / 2, float(campos.latitude.max()) + dy / 2]
    ar = (ext[1] - ext[0]) / max(ext[3] - ext[2], 1e-6)
    w = float(np.clip(3.2 * ar, 2.6, 5.5))
    # escala de color común a todos los paneles
    if simetrico:
        m = float(np.nanquantile(np.abs(campos.values), 0.99)); m = m if m > 0 else 1.0
        vmin, vmax = -m, m
    else:
        vmin, vmax = float(campos.quantile(0.01)), float(campos.quantile(0.99))
    kw = {"subplot_kw": {"projection": ccrs.PlateCarree()}} if True else {}
    fig, axs = plt.subplots(nfil, ncol, figsize=(w * ncol + 1.5, 3.1 * nfil + 1.2), squeeze=False, **kw)
    for k in range(nfil * ncol):
        i, j = divmod(k, ncol)
        ax = axs[i, j]
        if k >= n:
            ax.set_visible(False)
            continue
        if True:
            ax.set_extent(ext, crs=ccrs.PlateCarree())
            ax.coastlines(linewidth=0.6); ax.add_feature(cfeature.BORDERS, linewidth=0.4)
            gl = ax.gridlines(draw_labels=True, linewidth=0.2, alpha=0.5)
            gl.top_labels = gl.right_labels = False
            gl.left_labels = (j == 0); gl.bottom_labels = (i == nfil - 1)
            gl.xlabel_style = gl.ylabel_style = {"size": 7}
            tr = {"transform": ccrs.PlateCarree()}
        else:
            ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.tick_params(labelsize=7); tr = {}
        campo = campos.isel({campos.dims[0]: k})
        pm = ax.pcolormesh(campo.longitude, campo.latitude, campo.values, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto", **tr)
        ax.set_title(etiquetas[k], fontsize=9)
    fig.suptitle(titulo, fontsize=13, y=0.995)
    fig.colorbar(pm, ax=axs.ravel().tolist(), shrink=0.8, pad=0.02, label=etiqueta)
    plt.show()

---
### **<font color="DodgerBlue">Ejercicio: </font>**

Van a replicar este estudio ahora para climatología de altura geopotencial y para la climatología de la precipitación. Para esto, usen los datos de 

* https://github.com/Dinamica-del-Clima-2027-1/Sistemas_monzonicos/blob/main/data/hgt.mon.mean.nc
* https://github.com/Dinamica-del-Clima-2027-1/Sistemas_monzonicos/blob/main/data/era5_precipitation_monthly_climatology_1deg.nc

Recuerden que deben calcular:
- Climatología anual y anual regional
- Climatologías mensuales
- Anomalías 

---